In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install indic-nlp-library

In [3]:
!pip install emoji

  Using cached emoji-2.15.0-py3-none-any.whl.metadata (5.7 kB)
Using cached emoji-2.15.0-py3-none-any.whl (608 kB)


In [4]:
!pip install indicsyllabifier

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for indicsyllabifier: filename=indicsyllabifier-0.3-py3-none-any.whl size=6083 sha256=e23a09343d3cd5ac75139a1b70d2663c9377e2c144a8e824de6f62e3f4df448e
  Stored in directory: /root/.cache/pip/wheels/7d/0b/e3/9e55ec62ff351fe269680bc76449fa441a1e44ec3512d443d7
  Created wheel for silpa_common: filename=silpa_common-0.3-py3-none-any.whl size=8469 sha256=cc8818b1742ec4241a732e87ae82d3a8b7cf585301e3478190f7b0c364bbbf31
  Stored in directory: /root/.cache/pip/wheels/cf/d2/8d/3e359d137f6114538f7b9c6b06c1855a12050de481790f839c
Successfully built indicsyllabifier silpa_common


In [5]:
!pip install tokenizers

In [6]:
import pandas as pd

file_path_train = '/content/drive/MyDrive/Tulu Hope Speech 2026/Hope_FG/Finegrained_train_data.csv'

# Example: Read a CSV file into a pandas DataFrame
try:
    df_train = pd.read_csv(file_path_train)
    print(f"Successfully loaded df_train - shape: {df_train.shape}", "\n")
    print(df_train.head())

except FileNotFoundError:
    print(f"Error: The file {file_path} was not found. Check your path.")

file_path_dev = '/content/drive/MyDrive/Tulu Hope Speech 2026/Hope_FG/Finegrained_dev_data.csv'   #test_data_withoutlabelCG.csv
# Example: Read a CSV file into a pandas DataFrame
try:
    df_dev = pd.read_csv(file_path_dev)
    print(f"Successfully loaded df_dev - shape: {df_dev.shape}", "\n")
    print(df_dev.head())

except FileNotFoundError:
    print(f"Error: The file {file_path_dev} was not found. Check your path.")

file_path_test = '/content/drive/MyDrive/Tulu Hope Speech 2026/Hope_FG/Finegrained_test_data.csv'
# Example: Read a CSV file into a pandas DataFrame
try:
    df_test = pd.read_csv(file_path_test)
    print(f"Successfully loaded df_test - shape: {df_test.shape}")
    print(df_test.head())

except FileNotFoundError:
    print(f"Error: The file {file_path_test} was not found. Check your path.")


Successfully loaded df_train - shape: (3185, 2) 

                                                Text            Label
0                                       Lv uuu Arjun   inspiring hope
1                                    Super ajji papa   inspiring hope
2                    Great a original video undo plz     hopelessness
3  Miss malpadhe thovondhlle,,, thank you sir the...  optimistic hope
4             super Comedy . full movie link manpule     hopelessness
Successfully loaded df_dev - shape: (682, 2) 

                                                Text            Label
0                                Ha ha ha .. super..   inspiring hope
1  mitun rai n leppare ballitnd hindu virodhi n l...     hopelessness
2  Bhaari porlathnd.. nanath eddedde project bara...  optimistic hope
3                                      bari ede andu   realistic hope
4  SALMAN PinKHAN NA MOVIE-D YEDDE UNDU REVIEW MO...   inspiring hope
Successfully loaded df_test - shape: (683, 3)
     ID          

In [7]:
display(df_test)

,ID,Text,Label
0,3961,Heavy heavy enchina avasthe marreyy.,hopelessness
1,1064,"Dislike dhaye malpuvar,,,, support malpule nam...",optimistic hope
2,4805,Mulpa Nadiya sikkji,hopelessness
3,690,Must ista apune nikl comedy..super ya,optimistic hope
4,143,My name is annapa tulu movie upload manpule br...,hopelessness
...,...,...,...
678,2699,Ath tundala boreing apuji..,realistic hope
679,679,super comedy.... total entertainment,inspiring hope
680,4098,Team judge na comments seriously detonodu inch...,hopelessness
681,3241,Aravind bolerna overacted comedy films gethd d...,fading hope


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import regex as re
import string
import emoji
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.util import ngrams
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from indicnlp.tokenize import indic_tokenize
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report,precision_score, recall_score, f1_score

In [9]:
df_train = df_train.dropna(subset=["Text"])
df_train["text"] = df_train["Text"].astype(str)
print(df_train['text'])

df_dev = df_dev.dropna(subset=["Text"])
df_dev["text"] = df_dev["Text"].astype(str)
print(df_dev['text'])

df_test  = df_test.dropna(subset=["Text"])
df_test["text"]  = df_test["Text"].astype(str)
print(df_test['text'])

0                                            Lv uuu Arjun
1                                         Super ajji papa
2                         Great a original video undo plz
3       Miss malpadhe thovondhlle,,, thank you sir the...
4                  super Comedy . full movie link manpule
                              ...                        
3180                                       Super adu undu
3181          Umbe video views barre.g sullu panond ulle.
3182    super undu comedy.... Masth super expression u...
3183                                Full movie paadle plz
3184           Yaaan saithe Maree banji bene avondunduuuu
Name: text, Length: 3185, dtype: object
0                                    Ha ha ha .. super..
1      mitun rai n leppare ballitnd hindu virodhi n l...
2      Bhaari porlathnd.. nanath eddedde project bara...
3                                          bari ede andu
4      SALMAN PinKHAN NA MOVIE-D YEDDE UNDU REVIEW MO...
                             ...     

In [10]:
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [11]:
stop_words=set(stopwords.words("english"))
lemmatizer=WordNetLemmatizer()
stemmer = PorterStemmer()

In [12]:
def preprocess(text):
    if not isinstance(text, str):
        return text
    # Remove URLs
    text =re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove emails
    text =re.sub(r'\S+@\S+', '', text)
    # Remove emojis
    text = emoji.demojize(text, delimiters=(" ", " "))
    # Remove punctuations & numbers
    text=re.sub(r'\d+', '', text)
    # keeps english and kannada text
    text = re.sub(r'[^a-zA-Z\u0C80-\u0CFF\s]', '', text)
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    # Tokenize using indic_tokenize
    tokens=indic_tokenize.trivial_tokenize(text, lang='kn')
    return ' '.join(tokens)

df_train['processed_text'] = df_train['text'].apply(preprocess)
df_dev['processed_text'] = df_dev['text'].apply(preprocess)
df_test['processed_text'] = df_test['text'].apply(preprocess)

display(df_train[['processed_text']])
display(df_dev[['processed_text']])
display(df_test[['processed_text']])

,processed_text
0,Lv uuu Arjun
1,Super ajji papa
2,Great a original video undo plz
3,Miss malpadhe thovondhlle thank you sir thelpa...
4,super Comedy full movie link manpule
...,...
3180,Super adu undu
3181,Umbe video views barreg sullu panond ulle
3182,super undu comedy Masth super expression undu ...
3183,Full movie paadle plz


,processed_text
0,Ha ha ha super
1,mitun rai n leppare ballitnd hindu virodhi n l...
2,Bhaari porlathnd nanath eddedde project barad ...
3,bari ede andu
4,SALMAN PinKHAN NA MOVIED YEDDE UNDU REVIEW MOV...
...,...
677,superb tuluNada ratha prashmasha team
678,Yenk korolisuper comedytoo good
679,Tulu movies old strong Aussie team laka eppare...
680,Super bolar anna


,processed_text
0,Heavy heavy enchina avasthe marreyy
1,Dislike dhaye malpuvar support malpule nammakuleg
2,Mulpa Nadiya sikkji
3,Must ista apune nikl comedysuper ya
4,My name is annapa tulu movie upload manpule br...
...,...
678,Ath tundala boreing apuji
679,super comedy total entertainment
680,Team judge na comments seriously detonodu inch...
681,Aravind bolerna overacted comedy films gethd d...


#**word ngrams**

In [13]:
def generate_word_ngrams_joined(tokens, n):
    if not isinstance(tokens, list):
        return []
    return [' '.join(gram) for gram in ngrams(tokens, n)]

# Step 1: Convert processed text into tokens (list of words)
df_train['tokens'] = df_train['processed_text'].apply(lambda x: x.split() if isinstance(x, str) else [])
df_dev['tokens']  = df_dev['processed_text'].apply(lambda x: x.split() if isinstance(x, str) else [])
df_test['tokens']  = df_test['processed_text'].apply(lambda x: x.split() if isinstance(x, str) else [])

# Step 2: Generate word n-grams
for df in [df_train, df_dev, df_test]:
    df['word_unigrams_str'] = df['tokens'].apply(lambda tokens: generate_word_ngrams_joined(tokens, 1))
    df['word_bigrams_str']  = df['tokens'].apply(lambda tokens: generate_word_ngrams_joined(tokens, 2))
    df['word_trigrams_str'] = df['tokens'].apply(lambda tokens: generate_word_ngrams_joined(tokens, 3))

# Step 3: Display results
display(df_train[['processed_text', 'word_unigrams_str', 'word_bigrams_str', 'word_trigrams_str']])
display(df_dev[['processed_text', 'word_unigrams_str', 'word_bigrams_str', 'word_trigrams_str']])
display(df_test[['processed_text', 'word_unigrams_str', 'word_bigrams_str', 'word_trigrams_str']])

# ---- Merge unigrams, bigrams, trigrams into a single list ----
df_train['word_ngrams'] = (df_train['word_unigrams_str']+ df_train['word_bigrams_str']+ df_train['word_trigrams_str'])
df_dev['word_ngrams'] = (df_dev['word_unigrams_str']+ df_dev['word_bigrams_str']+ df_dev['word_trigrams_str'])
df_test['word_ngrams'] = (df_test['word_unigrams_str']+ df_test['word_bigrams_str']+ df_test['word_trigrams_str'])

display(df_train[['processed_text', 'word_ngrams']])

,processed_text,word_unigrams_str,word_bigrams_str,word_trigrams_str
0,Lv uuu Arjun,"[Lv, uuu, Arjun]","[Lv uuu, uuu Arjun]",[Lv uuu Arjun]
1,Super ajji papa,"[Super, ajji, papa]","[Super ajji, ajji papa]",[Super ajji papa]
2,Great a original video undo plz,"[Great, a, original, video, undo, plz]","[Great a, a original, original video, video un...","[Great a original, a original video, original ..."
3,Miss malpadhe thovondhlle thank you sir thelpa...,"[Miss, malpadhe, thovondhlle, thank, you, sir,...","[Miss malpadhe, malpadhe thovondhlle, thovondh...","[Miss malpadhe thovondhlle, malpadhe thovondhl..."
4,super Comedy full movie link manpule,"[super, Comedy, full, movie, link, manpule]","[super Comedy, Comedy full, full movie, movie ...","[super Comedy full, Comedy full movie, full mo..."
...,...,...,...,...
3180,Super adu undu,"[Super, adu, undu]","[Super adu, adu undu]",[Super adu undu]
3181,Umbe video views barreg sullu panond ulle,"[Umbe, video, views, barreg, sullu, panond, ulle]","[Umbe video, video views, views barreg, barreg...","[Umbe video views, video views barreg, views b..."
3182,super undu comedy Masth super expression undu ...,"[super, undu, comedy, Masth, super, expression...","[super undu, undu comedy, comedy Masth, Masth ...","[super undu comedy, undu comedy Masth, comedy ..."
3183,Full movie paadle plz,"[Full, movie, paadle, plz]","[Full movie, movie paadle, paadle plz]","[Full movie paadle, movie paadle plz]"


,processed_text,word_unigrams_str,word_bigrams_str,word_trigrams_str
0,Ha ha ha super,"[Ha, ha, ha, super]","[Ha ha, ha ha, ha super]","[Ha ha ha, ha ha super]"
1,mitun rai n leppare ballitnd hindu virodhi n l...,"[mitun, rai, n, leppare, ballitnd, hindu, viro...","[mitun rai, rai n, n leppare, leppare ballitnd...","[mitun rai n, rai n leppare, n leppare ballitn..."
2,Bhaari porlathnd nanath eddedde project barad ...,"[Bhaari, porlathnd, nanath, eddedde, project, ...","[Bhaari porlathnd, porlathnd nanath, nanath ed...","[Bhaari porlathnd nanath, porlathnd nanath edd..."
3,bari ede andu,"[bari, ede, andu]","[bari ede, ede andu]",[bari ede andu]
4,SALMAN PinKHAN NA MOVIED YEDDE UNDU REVIEW MOV...,"[SALMAN, PinKHAN, NA, MOVIED, YEDDE, UNDU, REV...","[SALMAN PinKHAN, PinKHAN NA, NA MOVIED, MOVIED...","[SALMAN PinKHAN NA, PinKHAN NA MOVIED, NA MOVI..."
...,...,...,...,...
677,superb tuluNada ratha prashmasha team,"[superb, tuluNada, ratha, prashmasha, team]","[superb tuluNada, tuluNada ratha, ratha prashm...","[superb tuluNada ratha, tuluNada ratha prashma..."
678,Yenk korolisuper comedytoo good,"[Yenk, korolisuper, comedytoo, good]","[Yenk korolisuper, korolisuper comedytoo, come...","[Yenk korolisuper comedytoo, korolisuper comed..."
679,Tulu movies old strong Aussie team laka eppare...,"[Tulu, movies, old, strong, Aussie, team, laka...","[Tulu movies, movies old, old strong, strong A...","[Tulu movies old, movies old strong, old stron..."
680,Super bolar anna,"[Super, bolar, anna]","[Super bolar, bolar anna]",[Super bolar anna]


,processed_text,word_unigrams_str,word_bigrams_str,word_trigrams_str
0,Heavy heavy enchina avasthe marreyy,"[Heavy, heavy, enchina, avasthe, marreyy]","[Heavy heavy, heavy enchina, enchina avasthe, ...","[Heavy heavy enchina, heavy enchina avasthe, e..."
1,Dislike dhaye malpuvar support malpule nammakuleg,"[Dislike, dhaye, malpuvar, support, malpule, n...","[Dislike dhaye, dhaye malpuvar, malpuvar suppo...","[Dislike dhaye malpuvar, dhaye malpuvar suppor..."
2,Mulpa Nadiya sikkji,"[Mulpa, Nadiya, sikkji]","[Mulpa Nadiya, Nadiya sikkji]",[Mulpa Nadiya sikkji]
3,Must ista apune nikl comedysuper ya,"[Must, ista, apune, nikl, comedysuper, ya]","[Must ista, ista apune, apune nikl, nikl comed...","[Must ista apune, ista apune nikl, apune nikl ..."
4,My name is annapa tulu movie upload manpule br...,"[My, name, is, annapa, tulu, movie, upload, ma...","[My name, name is, is annapa, annapa tulu, tul...","[My name is, name is annapa, is annapa tulu, a..."
...,...,...,...,...
678,Ath tundala boreing apuji,"[Ath, tundala, boreing, apuji]","[Ath tundala, tundala boreing, boreing apuji]","[Ath tundala boreing, tundala boreing apuji]"
679,super comedy total entertainment,"[super, comedy, total, entertainment]","[super comedy, comedy total, total entertainment]","[super comedy total, comedy total entertainment]"
680,Team judge na comments seriously detonodu inch...,"[Team, judge, na, comments, seriously, detonod...","[Team judge, judge na, na comments, comments s...","[Team judge na, judge na comments, na comments..."
681,Aravind bolerna overacted comedy films gethd d...,"[Aravind, bolerna, overacted, comedy, films, g...","[Aravind bolerna, bolerna overacted, overacted...","[Aravind bolerna overacted, bolerna overacted ..."


,processed_text,word_ngrams
0,Lv uuu Arjun,"[Lv, uuu, Arjun, Lv uuu, uuu Arjun, Lv uuu Arjun]"
1,Super ajji papa,"[Super, ajji, papa, Super ajji, ajji papa, Sup..."
2,Great a original video undo plz,"[Great, a, original, video, undo, plz, Great a..."
3,Miss malpadhe thovondhlle thank you sir thelpa...,"[Miss, malpadhe, thovondhlle, thank, you, sir,..."
4,super Comedy full movie link manpule,"[super, Comedy, full, movie, link, manpule, su..."
...,...,...
3180,Super adu undu,"[Super, adu, undu, Super adu, adu undu, Super ..."
3181,Umbe video views barreg sullu panond ulle,"[Umbe, video, views, barreg, sullu, panond, ul..."
3182,super undu comedy Masth super expression undu ...,"[super, undu, comedy, Masth, super, expression..."
3183,Full movie paadle plz,"[Full, movie, paadle, plz, Full movie, movie p..."


#**char ngrams**

In [14]:
def text_graphemes(text):
    if not isinstance(text, str):
        return []
    # normalize spaces
    text = text.strip()
    return re.findall(r'\X', text)   # get extended grapheme clusters (works for all scripts)

def char_ngrams(text, n):
    tokens = text_graphemes(text)
    return [''.join(g) for g in ngrams(tokens, n)]

df_train['char_unigrams'] = df_train['processed_text'].apply(lambda x: char_ngrams(x, 1))
df_train['char_bigrams']  = df_train['processed_text'].apply(lambda x: char_ngrams(x, 2))
df_train['char_trigrams'] = df_train['processed_text'].apply(lambda x: char_ngrams(x, 3))

df_dev['char_unigrams'] = df_dev['processed_text'].apply(lambda x: char_ngrams(x, 1))
df_dev['char_bigrams']  = df_dev['processed_text'].apply(lambda x: char_ngrams(x, 2))
df_dev['char_trigrams'] = df_dev['processed_text'].apply(lambda x: char_ngrams(x, 3))

df_test['char_unigrams'] = df_test['processed_text'].apply(lambda x: char_ngrams(x, 1))
df_test['char_bigrams']  = df_test['processed_text'].apply(lambda x: char_ngrams(x, 2))
df_test['char_trigrams'] = df_test['processed_text'].apply(lambda x: char_ngrams(x, 3))

display(df_train[['processed_text','char_unigrams','char_bigrams','char_trigrams']])
display(df_dev[['processed_text','char_unigrams','char_bigrams','char_trigrams']])
display(df_test[['processed_text','char_unigrams','char_bigrams','char_trigrams']])

def normalize_ngram(ngram):
    if not isinstance(ngram, str):
        return ""
    return unicodedata.normalize("NFC", ngram)

def merge_ngram_columns(row):
    unigrams = [normalize_ngram(t) for t in row.get("char_unigrams", []) if isinstance(t, str)]
    bigrams  = [normalize_ngram(t) for t in row.get("char_bigrams", []) if isinstance(t, str)]
    trigrams = [normalize_ngram(t) for t in row.get("char_trigrams", []) if isinstance(t, str)]

    return unigrams + bigrams + trigrams

df_train["char_ngrams"] = df_train.apply(merge_ngram_columns, axis=1)
df_dev["char_ngrams"]  = df_dev.apply(merge_ngram_columns, axis=1)
df_test["char_ngrams"]  = df_test.apply(merge_ngram_columns, axis=1)

display(df_train["char_ngrams"])

,processed_text,char_unigrams,char_bigrams,char_trigrams
0,Lv uuu Arjun,"[L, v, , u, u, u, , A, r, j, u, n]","[Lv, v , u, uu, uu, u , A, Ar, rj, ju, un]","[Lv , v u, uu, uuu, uu , u A, Ar, Arj, rju, ..."
1,Super ajji papa,"[S, u, p, e, r, , a, j, j, i, , p, a, p, a]","[Su, up, pe, er, r , a, aj, jj, ji, i , p, p...","[Sup, upe, per, er , r a, aj, ajj, jji, ji , ..."
2,Great a original video undo plz,"[G, r, e, a, t, , a, , o, r, i, g, i, n, a, ...","[Gr, re, ea, at, t , a, a , o, or, ri, ig, g...","[Gre, rea, eat, at , t a, a , a o, or, ori, ..."
3,Miss malpadhe thovondhlle thank you sir thelpa...,"[M, i, s, s, , m, a, l, p, a, d, h, e, , t, ...","[Mi, is, ss, s , m, ma, al, lp, pa, ad, dh, h...","[Mis, iss, ss , s m, ma, mal, alp, lpa, pad, ..."
4,super Comedy full movie link manpule,"[s, u, p, e, r, , C, o, m, e, d, y, , f, u, ...","[su, up, pe, er, r , C, Co, om, me, ed, dy, y...","[sup, upe, per, er , r C, Co, Com, ome, med, ..."
...,...,...,...,...
3180,Super adu undu,"[S, u, p, e, r, , a, d, u, , u, n, d, u]","[Su, up, pe, er, r , a, ad, du, u , u, un, n...","[Sup, upe, per, er , r a, ad, adu, du , u u, ..."
3181,Umbe video views barreg sullu panond ulle,"[U, m, b, e, , v, i, d, e, o, , v, i, e, w, ...","[Um, mb, be, e , v, vi, id, de, eo, o , v, v...","[Umb, mbe, be , e v, vi, vid, ide, deo, eo , ..."
3182,super undu comedy Masth super expression undu ...,"[s, u, p, e, r, , u, n, d, u, , c, o, m, e, ...","[su, up, pe, er, r , u, un, nd, du, u , c, c...","[sup, upe, per, er , r u, un, und, ndu, du , ..."
3183,Full movie paadle plz,"[F, u, l, l, , m, o, v, i, e, , p, a, a, d, ...","[Fu, ul, ll, l , m, mo, ov, vi, ie, e , p, p...","[Ful, ull, ll , l m, mo, mov, ovi, vie, ie , ..."


,processed_text,char_unigrams,char_bigrams,char_trigrams
0,Ha ha ha super,"[H, a, , h, a, , h, a, , s, u, p, e, r]","[Ha, a , h, ha, a , h, ha, a , s, su, up, p...","[Ha , a h, ha, ha , a h, ha, ha , a s, su, ..."
1,mitun rai n leppare ballitnd hindu virodhi n l...,"[m, i, t, u, n, , r, a, i, , n, , l, e, p, ...","[mi, it, tu, un, n , r, ra, ai, i , n, n , ...","[mit, itu, tun, un , n r, ra, rai, ai , i n, ..."
2,Bhaari porlathnd nanath eddedde project barad ...,"[B, h, a, a, r, i, , p, o, r, l, a, t, h, n, ...","[Bh, ha, aa, ar, ri, i , p, po, or, rl, la, a...","[Bha, haa, aar, ari, ri , i p, po, por, orl, ..."
3,bari ede andu,"[b, a, r, i, , e, d, e, , a, n, d, u]","[ba, ar, ri, i , e, ed, de, e , a, an, nd, du]","[bar, ari, ri , i e, ed, ede, de , e a, an, ..."
4,SALMAN PinKHAN NA MOVIED YEDDE UNDU REVIEW MOV...,"[S, A, L, M, A, N, , P, i, n, K, H, A, N, , ...","[SA, AL, LM, MA, AN, N , P, Pi, in, nK, KH, H...","[SAL, ALM, LMA, MAN, AN , N P, Pi, Pin, inK, ..."
...,...,...,...,...
677,superb tuluNada ratha prashmasha team,"[s, u, p, e, r, b, , t, u, l, u, N, a, d, a, ...","[su, up, pe, er, rb, b , t, tu, ul, lu, uN, N...","[sup, upe, per, erb, rb , b t, tu, tul, ulu, ..."
678,Yenk korolisuper comedytoo good,"[Y, e, n, k, , k, o, r, o, l, i, s, u, p, e, ...","[Ye, en, nk, k , k, ko, or, ro, ol, li, is, s...","[Yen, enk, nk , k k, ko, kor, oro, rol, oli, ..."
679,Tulu movies old strong Aussie team laka eppare...,"[T, u, l, u, , m, o, v, i, e, s, , o, l, d, ...","[Tu, ul, lu, u , m, mo, ov, vi, ie, es, s , ...","[Tul, ulu, lu , u m, mo, mov, ovi, vie, ies, ..."
680,Super bolar anna,"[S, u, p, e, r, , b, o, l, a, r, , a, n, n, a]","[Su, up, pe, er, r , b, bo, ol, la, ar, r , ...","[Sup, upe, per, er , r b, bo, bol, ola, lar, ..."


,processed_text,char_unigrams,char_bigrams,char_trigrams
0,Heavy heavy enchina avasthe marreyy,"[H, e, a, v, y, , h, e, a, v, y, , e, n, c, ...","[He, ea, av, vy, y , h, he, ea, av, vy, y , ...","[Hea, eav, avy, vy , y h, he, hea, eav, avy, ..."
1,Dislike dhaye malpuvar support malpule nammakuleg,"[D, i, s, l, i, k, e, , d, h, a, y, e, , m, ...","[Di, is, sl, li, ik, ke, e , d, dh, ha, ay, y...","[Dis, isl, sli, lik, ike, ke , e d, dh, dha, ..."
2,Mulpa Nadiya sikkji,"[M, u, l, p, a, , N, a, d, i, y, a, , s, i, ...","[Mu, ul, lp, pa, a , N, Na, ad, di, iy, ya, a...","[Mul, ulp, lpa, pa , a N, Na, Nad, adi, diy, ..."
3,Must ista apune nikl comedysuper ya,"[M, u, s, t, , i, s, t, a, , a, p, u, n, e, ...","[Mu, us, st, t , i, is, st, ta, a , a, ap, p...","[Mus, ust, st , t i, is, ist, sta, ta , a a, ..."
4,My name is annapa tulu movie upload manpule br...,"[M, y, , n, a, m, e, , i, s, , a, n, n, a, ...","[My, y , n, na, am, me, e , i, is, s , a, a...","[My , y n, na, nam, ame, me , e i, is, is , ..."
...,...,...,...,...
678,Ath tundala boreing apuji,"[A, t, h, , t, u, n, d, a, l, a, , b, o, r, ...","[At, th, h , t, tu, un, nd, da, al, la, a , ...","[Ath, th , h t, tu, tun, und, nda, dal, ala, ..."
679,super comedy total entertainment,"[s, u, p, e, r, , c, o, m, e, d, y, , t, o, ...","[su, up, pe, er, r , c, co, om, me, ed, dy, y...","[sup, upe, per, er , r c, co, com, ome, med, ..."
680,Team judge na comments seriously detonodu inch...,"[T, e, a, m, , j, u, d, g, e, , n, a, , c, ...","[Te, ea, am, m , j, ju, ud, dg, ge, e , n, n...","[Tea, eam, am , m j, ju, jud, udg, dge, ge , ..."
681,Aravind bolerna overacted comedy films gethd d...,"[A, r, a, v, i, n, d, , b, o, l, e, r, n, a, ...","[Ar, ra, av, vi, in, nd, d , b, bo, ol, le, e...","[Ara, rav, avi, vin, ind, nd , d b, bo, bol, ..."


,char_ngrams
0,"[L, v, , u, u, u, , A, r, j, u, n, Lv, v , ..."
1,"[S, u, p, e, r, , a, j, j, i, , p, a, p, a, ..."
2,"[G, r, e, a, t, , a, , o, r, i, g, i, n, a, ..."
3,"[M, i, s, s, , m, a, l, p, a, d, h, e, , t, ..."
4,"[s, u, p, e, r, , C, o, m, e, d, y, , f, u, ..."
...,...
3180,"[S, u, p, e, r, , a, d, u, , u, n, d, u, Su,..."
3181,"[U, m, b, e, , v, i, d, e, o, , v, i, e, w, ..."
3182,"[s, u, p, e, r, , u, n, d, u, , c, o, m, e, ..."
3183,"[F, u, l, l, , m, o, v, i, e, , p, a, a, d, ..."


In [15]:
from indicsyllabifier.core import Syllabalizer
import re

syllabifier = Syllabalizer()

# very simple English syllable splitter (rule-based)
def english_syllables(word):
    word = word.lower()
    if not word.isalpha():
        return []
    # split based on vowel groups
    return re.findall(r'[^aeiouy]*[aeiouy]+(?:[^aeiouy]*)', word)

# detect Kannada characters
def contains_kannada(text):
    return bool(re.search(r'[\u0C80-\u0CFF]', text))

def get_syllables_bilingual(text):
    if not isinstance(text, str) or text.strip() == "":
      return []

    syllables = []
    tokens = text.split()
    for token in tokens:
      if contains_kannada(token):
        try:
          syls = syllabifier.syllabify(token)
          syllables.extend(syls)
        except:
          pass

      elif token.isascii():
        syls = english_syllables(token)
        syllables.extend(syls)
    return syllables

def syllable_ngrams(syllables, n):
    return [''.join(syllables[i:i+n]) for i in range(len(syllables) - n + 1)]


df_train['syllables'] = df_train['processed_text'].apply(get_syllables_bilingual)
df_train['syllable_bigrams'] = df_train['syllables'].apply(lambda x: syllable_ngrams(x, 2))
df_train['syllable_trigrams'] = df_train['syllables'].apply(lambda x: syllable_ngrams(x, 3))

df_dev['syllables'] = df_dev['processed_text'].apply(get_syllables_bilingual)
df_dev['syllable_bigrams'] = df_dev['syllables'].apply(lambda x: syllable_ngrams(x, 2))
df_dev['syllable_trigrams'] = df_dev['syllables'].apply(lambda x: syllable_ngrams(x, 3))

df_test['syllables'] = df_test['processed_text'].apply(get_syllables_bilingual)
df_test['syllable_bigrams'] = df_test['syllables'].apply(lambda x: syllable_ngrams(x, 2))
df_test['syllable_trigrams'] = df_test['syllables'].apply(lambda x: syllable_ngrams(x, 3))

display(df_train[['processed_text', 'syllables', 'syllable_bigrams', 'syllable_trigrams']])
display(df_dev[['processed_text', 'syllables', 'syllable_bigrams', 'syllable_trigrams']])
display(df_test[['processed_text', 'syllables', 'syllable_bigrams', 'syllable_trigrams']])


def normalize_syllable_ngram(ngram):
    if not isinstance(ngram, str):
        return ""
    return unicodedata.normalize("NFC", ngram)

def merge_syllable_ngram_columns(row):
    unigrams = [normalize_syllable_ngram(t) for t in row.get("syllables", []) if isinstance(t, str)]
    bigrams  = [normalize_syllable_ngram(t) for t in row.get("syllable_bigrams", []) if isinstance(t, str)]
    trigrams = [normalize_syllable_ngram(t) for t in row.get("syllable_trigrams", []) if isinstance(t, str)]
    return unigrams + bigrams + trigrams

df_train["syllable_ngrams"] = df_train.apply(merge_syllable_ngram_columns, axis=1)
df_dev["syllable_ngrams"]  = df_dev.apply(merge_syllable_ngram_columns, axis=1)
df_test["syllable_ngrams"]  = df_test.apply(merge_syllable_ngram_columns, axis=1)

display(df_train["syllable_ngrams"])

,processed_text,syllables,syllable_bigrams,syllable_trigrams
0,Lv uuu Arjun,"[uuu, arj, un]","[uuuarj, arjun]",[uuuarjun]
1,Super ajji papa,"[sup, er, ajj, i, pap, a]","[super, erajj, ajji, ipap, papa]","[superajj, erajji, ajjipap, ipapa]"
2,Great a original video undo plz,"[great, a, or, ig, in, al, vid, eo, und, o]","[greata, aor, orig, igin, inal, alvid, video, ...","[greataor, aorig, origin, iginal, inalvid, alv..."
3,Miss malpadhe thovondhlle thank you sir thelpa...,"[miss, malp, adh, e, thov, ondhll, e, thank, y...","[missmalp, malpadh, adhe, ethov, thovondhll, o...","[missmalpadh, malpadhe, adhethov, ethovondhll,..."
4,super Comedy full movie link manpule,"[sup, er, com, ed, y, full, mov, ie, link, man...","[super, ercom, comed, edy, yfull, fullmov, mov...","[supercom, ercomed, comedy, edyfull, yfullmov,..."
...,...,...,...,...
3180,Super adu undu,"[sup, er, ad, u, und, u]","[super, erad, adu, uund, undu]","[superad, eradu, aduund, uundu]"
3181,Umbe video views barreg sullu panond ulle,"[umb, e, vid, eo, views, barr, eg, sull, u, pa...","[umbe, evid, video, eoviews, viewsbarr, barreg...","[umbevid, evideo, videoviews, eoviewsbarr, vie..."
3182,super undu comedy Masth super expression undu ...,"[sup, er, und, u, com, ed, y, masth, sup, er, ...","[super, erund, undu, ucom, comed, edy, ymasth,...","[superund, erundu, unducom, ucomed, comedy, ed..."
3183,Full movie paadle plz,"[full, mov, ie, paadl, e]","[fullmov, movie, iepaadl, paadle]","[fullmovie, moviepaadl, iepaadle]"


,processed_text,syllables,syllable_bigrams,syllable_trigrams
0,Ha ha ha super,"[ha, ha, ha, sup, er]","[haha, haha, hasup, super]","[hahaha, hahasup, hasuper]"
1,mitun rai n leppare ballitnd hindu virodhi n l...,"[mit, un, rai, lepp, ar, e, ball, itnd, hind, ...","[mitun, unrai, railepp, leppar, are, eball, ba...","[mitunrai, unrailepp, raileppar, leppare, areb..."
2,Bhaari porlathnd nanath eddedde project barad ...,"[bhaar, i, porl, athnd, nan, ath, edd, edd, e,...","[bhaari, iporl, porlathnd, athndnan, nanath, a...","[bhaariporl, iporlathnd, porlathndnan, athndna..."
3,bari ede andu,"[bar, i, ed, e, and, u]","[bari, ied, ede, eand, andu]","[baried, iede, edeand, eandu]"
4,SALMAN PinKHAN NA MOVIED YEDDE UNDU REVIEW MOV...,"[salm, an, pinkh, an, na, mov, ied, yedd, e, u...","[salman, anpinkh, pinkhan, anna, namov, movied...","[salmanpinkh, anpinkhan, pinkhanna, annamov, n..."
...,...,...,...,...
677,superb tuluNada ratha prashmasha team,"[sup, erb, tul, un, ad, a, rath, a, prashm, as...","[superb, erbtul, tulun, unad, ada, arath, rath...","[superbtul, erbtulun, tulunad, unada, adarath,..."
678,Yenk korolisuper comedytoo good,"[yenk, kor, ol, is, up, er, com, ed, yt, oo, g...","[yenkkor, korol, olis, isup, uper, ercom, come...","[yenkkorol, korolis, olisup, isuper, upercom, ..."
679,Tulu movies old strong Aussie team laka eppare...,"[tul, u, mov, ies, old, strong, auss, ie, team...","[tulu, umov, movies, iesold, oldstrong, strong...","[tulumov, umovies, moviesold, iesoldstrong, ol..."
680,Super bolar anna,"[sup, er, bol, ar, ann, a]","[super, erbol, bolar, arann, anna]","[superbol, erbolar, bolarann, aranna]"


,processed_text,syllables,syllable_bigrams,syllable_trigrams
0,Heavy heavy enchina avasthe marreyy,"[heav, y, heav, y, ench, in, a, av, asth, e, m...","[heavy, yheav, heavy, yench, enchin, ina, aav,...","[heavyheav, yheavy, heavyench, yenchin, enchin..."
1,Dislike dhaye malpuvar support malpule nammakuleg,"[disl, ik, e, dhaye, malp, uv, ar, supp, ort, ...","[dislik, ike, edhaye, dhayemalp, malpuv, uvar,...","[dislike, ikedhaye, edhayemalp, dhayemalpuv, m..."
2,Mulpa Nadiya sikkji,"[mulp, a, nad, iya, sikkj, i]","[mulpa, anad, nadiya, iyasikkj, sikkji]","[mulpanad, anadiya, nadiyasikkj, iyasikkji]"
3,Must ista apune nikl comedysuper ya,"[must, ist, a, ap, un, e, nikl, com, ed, ys, u...","[mustist, ista, aap, apun, une, enikl, niklcom...","[mustista, istaap, aapun, apune, unenikl, enik..."
4,My name is annapa tulu movie upload manpule br...,"[my, nam, e, is, ann, ap, a, tul, u, mov, ie, ...","[mynam, name, eis, isann, annap, apa, atul, tu...","[myname, nameis, eisann, isannap, annapa, apat..."
...,...,...,...,...
678,Ath tundala boreing apuji,"[ath, tund, al, a, bor, eing, ap, uj, i]","[athtund, tundal, ala, abor, boreing, eingap, ...","[athtundal, tundala, alabor, aboreing, boreing..."
679,super comedy total entertainment,"[sup, er, com, ed, y, tot, al, ent, ert, ainm,...","[super, ercom, comed, edy, ytot, total, alent,...","[supercom, ercomed, comedy, edytot, ytotal, to..."
680,Team judge na comments seriously detonodu inch...,"[team, judg, e, na, comm, ents, ser, iousl, y,...","[teamjudg, judge, ena, nacomm, comments, entss...","[teamjudge, judgena, enacomm, nacomments, comm..."
681,Aravind bolerna overacted comedy films gethd d...,"[ar, av, ind, bol, ern, a, ov, er, act, ed, co...","[arav, avind, indbol, bolern, erna, aov, over,...","[aravind, avindbol, indbolern, bolerna, ernaov..."


,syllable_ngrams
0,"[uuu, arj, un, uuuarj, arjun, uuuarjun]"
1,"[sup, er, ajj, i, pap, a, super, erajj, ajji, ..."
2,"[great, a, or, ig, in, al, vid, eo, und, o, gr..."
3,"[miss, malp, adh, e, thov, ondhll, e, thank, y..."
4,"[sup, er, com, ed, y, full, mov, ie, link, man..."
...,...
3180,"[sup, er, ad, u, und, u, super, erad, adu, uun..."
3181,"[umb, e, vid, eo, views, barr, eg, sull, u, pa..."
3182,"[sup, er, und, u, com, ed, y, masth, sup, er, ..."
3183,"[full, mov, ie, paadl, e, fullmov, movie, iepa..."


#**sub words**

In [16]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import re

def contains_kannada(text):
    return bool(re.search(r'[\u0C80-\u0CFF]', text))

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(vocab_size=12000,min_frequency=2,special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"])
tokenizer.train_from_iterator(df_train["processed_text"].astype(str).tolist(),trainer=trainer)

def extract_bilingual_bpe_tokens(text_series, tokenizer):
    return text_series.apply(lambda x: tokenizer.encode(x).tokens if isinstance(x, str) else [])

df_train["subwords"] = extract_bilingual_bpe_tokens(df_train["processed_text"], tokenizer)
df_dev["subwords"] = extract_bilingual_bpe_tokens(df_dev["processed_text"], tokenizer)
df_test["subwords"] = extract_bilingual_bpe_tokens(df_test["processed_text"], tokenizer)

df_train["subword_text"] = df_train["subwords"].apply(lambda x: " ".join(x))
df_dev["subword_text"]  = df_dev["subwords"].apply(lambda x: " ".join(x))
df_test["subword_text"]  = df_test["subwords"].apply(lambda x: " ".join(x))

display(df_train[["processed_text", "subwords"]])
display(df_dev[["processed_text", "subwords"]])
display(df_test[["processed_text", "subwords"]])

,processed_text,subwords
0,Lv uuu Arjun,"[Lv, uuu, Arjun]"
1,Super ajji papa,"[Super, ajji, papa]"
2,Great a original video undo plz,"[Great, a, ori, ginal, video, undo, plz]"
3,Miss malpadhe thovondhlle thank you sir thelpa...,"[Miss, malpad, he, tho, vondh, lle, thank, you..."
4,super Comedy full movie link manpule,"[super, Comedy, full, movie, link, manpule]"
...,...,...
3180,Super adu undu,"[Super, adu, undu]"
3181,Umbe video views barreg sullu panond ulle,"[U, mbe, video, views, bar, reg, sul, lu, pano..."
3182,super undu comedy Masth super expression undu ...,"[super, undu, comedy, Masth, super, expression..."
3183,Full movie paadle plz,"[Full, movie, paadle, plz]"


,processed_text,subwords
0,Ha ha ha super,"[Ha, ha, ha, super]"
1,mitun rai n leppare ballitnd hindu virodhi n l...,"[mi, tu, n, rai, n, le, pp, are, ballitnd, hi,..."
2,Bhaari porlathnd nanath eddedde project barad ...,"[Bha, ari, porla, thnd, nana, th, ed, d, edde,..."
3,bari ede andu,"[bari, ede, andu]"
4,SALMAN PinKHAN NA MOVIED YEDDE UNDU REVIEW MOV...,"[S, AL, M, AN, P, in, K, H, AN, NA, M, O, VI, ..."
...,...,...
677,superb tuluNada ratha prashmasha team,"[superb, tulu, N, ada, ra, tha, prash, ma, sha..."
678,Yenk korolisuper comedytoo good,"[Yenk, kor, oli, super, comedy, too, good]"
679,Tulu movies old strong Aussie team laka eppare...,"[Tulu, movies, old, st, r, ong, A, u, ss, i, e..."
680,Super bolar anna,"[Super, bolar, anna]"


,processed_text,subwords
0,Heavy heavy enchina avasthe marreyy,"[He, avy, he, avy, enchina, avasthe, marre, yy]"
1,Dislike dhaye malpuvar support malpule nammakuleg,"[Dislike, dha, ye, malpuvar, support, malpule,..."
2,Mulpa Nadiya sikkji,"[Mul, pa, N, adi, ya, s, ikk, ji]"
3,Must ista apune nikl comedysuper ya,"[Mu, st, ista, apu, ne, nik, l, comedy, super,..."
4,My name is annapa tulu movie upload manpule br...,"[My, name, is, anna, pa, tulu, movie, upload, ..."
...,...,...
678,Ath tundala boreing apuji,"[A, th, tundala, boreing, apuji]"
679,super comedy total entertainment,"[super, comedy, to, tal, entertainment]"
680,Team judge na comments seriously detonodu inch...,"[Team, judge, na, comments, seriously, deton, ..."
681,Aravind bolerna overacted comedy films gethd d...,"[Aravind, bol, erna, over, act, ed, comedy, fi..."


In [17]:
def merge_all_features(row):

    word_feats = row.get("word_ngrams", [])
    char_feats = row.get("char_ngrams", [])
    syll_feats = row.get("syllable_ngrams", [])
    subword_feats = row.get("subwords", [])

    merged = []

    if isinstance(word_feats, list):
        merged.extend(word_feats)

    if isinstance(char_feats, list):
        merged.extend(char_feats)

    if isinstance(syll_feats, list):
        merged.extend(syll_feats)

    if isinstance(subword_feats, list):
        merged.extend(subword_feats)

    return merged

df_train["all_features"] = df_train.apply(merge_all_features, axis=1)
df_dev["all_features"]  = df_dev.apply(merge_all_features, axis=1)
df_test["all_features"]  = df_test.apply(merge_all_features, axis=1)


display(df_train[["processed_text", "all_features"]])

,processed_text,all_features
0,Lv uuu Arjun,"[Lv, uuu, Arjun, Lv uuu, uuu Arjun, Lv uuu Arj..."
1,Super ajji papa,"[Super, ajji, papa, Super ajji, ajji papa, Sup..."
2,Great a original video undo plz,"[Great, a, original, video, undo, plz, Great a..."
3,Miss malpadhe thovondhlle thank you sir thelpa...,"[Miss, malpadhe, thovondhlle, thank, you, sir,..."
4,super Comedy full movie link manpule,"[super, Comedy, full, movie, link, manpule, su..."
...,...,...
3180,Super adu undu,"[Super, adu, undu, Super adu, adu undu, Super ..."
3181,Umbe video views barreg sullu panond ulle,"[Umbe, video, views, barreg, sullu, panond, ul..."
3182,super undu comedy Masth super expression undu ...,"[super, undu, comedy, Masth, super, expression..."
3183,Full movie paadle plz,"[Full, movie, paadle, plz, Full movie, movie p..."


In [18]:
df_train["all_features_text"] = df_train["all_features"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
df_dev["all_features_text"] = df_dev["all_features"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
df_test["all_features_text"] = df_test["all_features"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train["all_features_text"])
X_dev_tfidf = tfidf_vectorizer.transform(df_dev["all_features_text"])
X_test_tfidf = tfidf_vectorizer.transform(df_test["all_features_text"])

feature_names = tfidf_vectorizer.get_feature_names_out()

print("Total TF-IDF features:", len(feature_names))
print("First 50 features:", feature_names[:50])


Total TF-IDF features: 45729
First 50 features: ['_e' '_eಎ' 'aa' 'aaa' 'aaaa' 'aaaaa' 'aaaaaa' 'aaaaaaa' 'aaaaaaaa'
 'aaaaaaaafac' 'aaaaaaaafacew' 'aaaaaaasakk' 'aaaaaaasakkath'
 'aaaaaaasanth' 'aaaaaaasanthuuu' 'aaaaanch' 'aaaaanchor' 'aaaaar'
 'aaaaareall' 'aaaaareally' 'aaaadad' 'aaaadada' 'aaaajj' 'aaaajji'
 'aaaajjibhayank' 'aaaam' 'aaaaman' 'aaaamanag' 'aaaand' 'aaaandthudh'
 'aaaarakk' 'aaaarakkil' 'aaac' 'aaacut' 'aaacute' 'aaadial' 'aaafad'
 'aaafading' 'aaagood' 'aaaij' 'aaaiji' 'aaaijimast' 'aaall' 'aaallu'
 'aaand' 'aaanda' 'aaandall' 'aaandann' 'aaandath' 'aaandmin']


In [19]:
tfidf_df_train = pd.DataFrame(X_train_tfidf.toarray(),columns=feature_names)
tfidf_df_dev = pd.DataFrame(X_dev_tfidf.toarray(),columns=feature_names)
tfidf_df_test = pd.DataFrame(X_test_tfidf.toarray(),columns=feature_names)


display(tfidf_df_train)
display(tfidf_df_dev)
display(tfidf_df_test)

,_e,_eಎ,aa,aaa,aaaa,aaaaa,aaaaaa,aaaaaaa,aaaaaaaa,aaaaaaaafac,...,ಸವ,ಸವಪ,ಸಸ,ಸಹ,ಸಹಜ,ಹಜ,ಹಜವ,ಹತ,ಹಳ,ಹಹ
0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3180,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3181,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3182,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3183,0.0,0.0,0.038398,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,_e,_eಎ,aa,aaa,aaaa,aaaaa,aaaaaa,aaaaaaa,aaaaaaaa,aaaaaaaafac,...,ಸವ,ಸವಪ,ಸಸ,ಸಹ,ಸಹಜ,ಹಜ,ಹಜವ,ಹತ,ಹಳ,ಹಹ
0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.056697,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
678,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
679,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
680,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,_e,_eಎ,aa,aaa,aaaa,aaaaa,aaaaaa,aaaaaaa,aaaaaaaa,aaaaaaaafac,...,ಸವ,ಸವಪ,ಸಸ,ಸಹ,ಸಹಜ,ಹಜ,ಹಜವ,ಹತ,ಹಳ,ಹಹ
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
678,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
679,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
680,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
681,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
# Voting Classifier
models = {
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

estimators = [(name, model) for name, model in models.items()]
voting_clf = VotingClassifier(estimators=estimators, voting='soft')

# Train
voting_clf.fit(X_train_tfidf, df_train['Label'])

# -------------------------
# DEV Evaluation
# -------------------------
y_dev_pred = voting_clf.predict(X_dev_tfidf)

print("DEV RESULTS")
print("Accuracy:", accuracy_score(df_dev['Label'], y_dev_pred))
print("Macro F1:", f1_score(df_dev['Label'], y_dev_pred, average='macro'))
print(classification_report(df_dev['Label'], y_dev_pred, zero_division=0))

# -------------------------
# TEST Evaluation
# -------------------------
y_test_pred = voting_clf.predict(X_test_tfidf)

print("\nTEST RESULTS")
print("Accuracy:", accuracy_score(df_test['Label'], y_test_pred))
print("Macro F1:", f1_score(df_test['Label'], y_test_pred, average='macro'))
print(classification_report(df_test['Label'], y_test_pred, zero_division=0))

DEV RESULTS
Accuracy: 0.40175953079178883
Macro F1: 0.3068316199741739
                 precision    recall  f1-score   support

    fading hope       0.18      0.12      0.14        51
   hopelessness       0.42      0.43      0.43       200
 inspiring hope       0.50      0.61      0.55       242
optimistic hope       0.29      0.23      0.26        81
 realistic hope       0.18      0.14      0.16       108

       accuracy                           0.40       682
      macro avg       0.31      0.31      0.31       682
   weighted avg       0.38      0.40      0.39       682


TEST RESULTS
Accuracy: 0.383601756954612
Macro F1: 0.2887753362264986
                 precision    recall  f1-score   support

    fading hope       0.10      0.08      0.09        50
   hopelessness       0.42      0.46      0.44       201
 inspiring hope       0.49      0.54      0.51       242
optimistic hope       0.25      0.20      0.22        82
 realistic hope       0.20      0.17      0.18       108